## Gradient-weighted Class Activation Mapping

In [ ]:
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from datasets import load_dataset
from torchvision.transforms.functional import to_pil_image
from IPython.display import display
from PIL import Image

from utils.hf_utils import ImageDatasetWithLabels

In [ ]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def register_hooks(model):
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False

    activations = {}
    gradients = {}

    target_layer = None
    for _, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            target_layer = module

    def forward_hook(module, input, output):
        activations["value"] = output
        output.register_hook(lambda grad: gradients.update({"value": grad}))

    forward_handle = target_layer.register_forward_hook(forward_hook)
    return activations, gradients, forward_handle

### VGG

In [ ]:
imagenet_model = torchvision.models.vgg16(pretrained=True)
imagenet_model = imagenet_model.to(device)

In [ ]:
hf_dataset = load_dataset("ILSVRC/imagenet-1k", split="train")
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
transform = transforms.Compose([
    transforms.RandomResizedCrop((224, 224)),
	transforms.ToTensor(),
    transforms.Normalize(
		mean=mean,
		std=std,
	),
])
imagenet_dataset = ImageDatasetWithLabels(hf_dataset, transform)

In [ ]:
random_idx = torch.randint(len(imagenet_dataset), (1,)).item()
imagenet_image, _ = imagenet_dataset[random_idx]
imagenet_image = imagenet_image.to(device)

imagenet_unnormalized = imagenet_image.squeeze(0).detach().cpu()
imagenet_unnormalized = (imagenet_unnormalized * std + mean).clamp(0, 1)
imagenet_pil_image = to_pil_image(imagenet_unnormalized).convert("RGB")

display(imagenet_pil_image)

In [ ]:
activations, gradients, forward_handle = register_hooks(imagenet_model)
pred = imagenet_model(imagenet_image.unsqueeze(0))
class_idx = pred.argmax(dim=1).item()
imagenet_model.zero_grad()
pred[0, class_idx].backward()

feature_maps = activations["value"]
grads = gradients["value"]
weights = grads.mean(dim=(2, 3), keepdim=True)
cam = (weights * feature_maps).sum(dim=1)
cam = F.relu(cam)
cam_min = cam.amin(dim=(1, 2), keepdim=True)
cam_max = cam.amax(dim=(1, 2), keepdim=True)

cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)
heatmap = F.interpolate(
    cam.unsqueeze(1),
    size=(224, 224),
    mode="bilinear",
    align_corners=False
)

imagenet_unnormalized = imagenet_image.squeeze(0).detach().cpu()
imagenet_unnormalized = (imagenet_unnormalized * std + mean).clamp(0, 1)
imagenet_pil_image = to_pil_image(imagenet_unnormalized).convert("RGB")

heatmap = heatmap[0, 0].detach().cpu().numpy()

heatmap_colored = cm.jet(heatmap)[:, :, :3]
heatmap_colored = (heatmap_colored * 255).astype(np.uint8)

heatmap_pil = Image.fromarray(heatmap_colored)

overlay = Image.blend(
    imagenet_pil_image,
    heatmap_pil,
    alpha=0.4
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(imagenet_pil_image)
axes[0].set_title("Image")

axes[1].imshow(heatmap_pil)
axes[1].set_title("Gradient-based Localizations")

axes[2].imshow(overlay)
axes[2].set_title("Grad-CAM")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()